# Notebook 03 — Retrained Model: Closing the Train/Inference Gap (Way 3)

## Problem
NB01 trains on bare SQL queries; NB02 infers on full URL paths.
Marketplace URL char-trigrams overlap with SQL patterns → 5,666 FPs (RF).

## Solution — Way 3
Train **and** infer on extracted query parameter values only.

## Fixes applied
- **Benign pool**: loads query values saved by NB02 (not raw URLs — previous
  version extracted zero pseudo-negatives because all URLs were path-only)
- **Synthetic negatives at scale**: ~964 entries covering IDs, page numbers,
  sort/filter, categories, locations, search terms — replaces 64-entry set
- **Threshold search corrected** (HIGH→LOW), T_high > T_low enforced
- Leakage-proof split-first pipeline; 5-fold CV; PR-AUC; 50+50 benchmark


## 1. Imports & Setup

In [3]:
import pandas as pd
import numpy as np
import joblib, os, re, time, json, urllib.parse, warnings
warnings.filterwarnings('ignore')

from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score, precision_recall_curve
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

for d in ['results/models', 'results/figures', 'results/metrics']:
    os.makedirs(d, exist_ok=True)

print('Setup complete.')
import sklearn
print(sklearn.__version__)


Setup complete.
1.4.2


## 2. Core Functions

In [2]:
SYMBOLS = ["'",'"',";","--","#","/*","*/","*","+","|","(",")",">","<","\\","/","="]

def extract_symbol_frequencies(text):
    c = Counter()
    for sym in SYMBOLS:
        c[sym] = str(text).count(sym)
    return c

def build_symbol_matrix(queries):
    rows = [extract_symbol_frequencies(q) for q in queries]
    return csr_matrix(pd.DataFrame(rows).fillna(0).values)

def extract_query_values(url):
    """Return joined query param VALUES, or None if no query string."""
    parsed = urllib.parse.urlparse(url)
    params = urllib.parse.parse_qs(parsed.query, keep_blank_values=False)
    values = [urllib.parse.unquote(v).strip()
              for vlist in params.values() for v in vlist
              if urllib.parse.unquote(v).strip()]
    return ' '.join(values) if values else None

TEMPLATES = [
    "/product?id={}",        "/search?q={}",
    "/login?username={}",     "/items?filter={}",
    "/page?param={}",         "/details?id={}",
    "/index.php?search={}",   "/admin/content?username={}",
    "/user?filter={}",        "/products?id={}",
]

print('extract_query_values() sanity checks:')
for url, lbl in [
    ("/search?q=laptop&page=2",                       "benign search"),
    ("/product?id=1' UNION SELECT null FROM users--",  "SQLi"),
    ("/index.php/ksa/all/ad/18797659",                 "path-only → None"),
    ("/details?id=42&sort=desc",                       "benign params"),
]:
    print(f'  [{lbl}] → {repr(extract_query_values(url))}')


extract_query_values() sanity checks:
  [benign search] → 'laptop 2'
  [SQLi] → "1' UNION SELECT null FROM users--"
  [path-only → None] → None
  [benign params] → '42 desc'


## 3. Positive Examples — SQLi Attacks

In [3]:
df_clean     = pd.read_csv('../datasets/Cleaned_SQL_Dataset.csv')
sqli_queries = df_clean[df_clean['Label'] == 1]['Query'].tolist()

positives = []
for i, q in enumerate(sqli_queries):
    tmpl = TEMPLATES[i % len(TEMPLATES)]
    url  = tmpl.format(urllib.parse.quote(str(q), safe="'\"=;-/*+()|<>\\#@ "))
    val  = extract_query_values(url)
    if val:
        positives.append(val)

print(f'SQLi queries in       : {len(sqli_queries):,}')
print(f'Positive examples out : {len(positives):,}')
print()
print('Sample positives (what the model sees):')
for p in positives[:4]:
    print(f"  '{p[:90]}'")


SQLi queries in       : 11,387
Positive examples out : 11,383

Sample positives (what the model sees):
  '" or pg_sleep  (  __TIME__  )  --'
  'create user name identified by pass123 temporary tablespace temp default tablespace users;'
  'AND 1  =  utl_inaddr.get_host_address   (    (   SELECT DISTINCT  (  table_name  )   FROM '
  'select * from users where id  =  '1' or @ @1  =  1 union select 1,version  (    )   -- 1''


## 4. Negative Examples

**Source 1 — pseudo-labeled from NB02** (RF score < 0.05, query values only).
NB02 now saves extracted query values directly — no more None-extraction problem.

**Source 2 — synthetic negatives** (~964 entries).
Covers numeric IDs, page numbers, sort/filter, categories, locations, search terms.
Replaces the 64-entry curated set that was insufficient for the model.


In [4]:
# ── Source 1: pseudo-labeled query values from NB02 ──────────────
hc_df            = pd.read_csv('results/benign_pool/02_hc_benign_qvalues.csv')
pseudo_negatives = hc_df['QueryValues'].dropna().tolist()
print(f'Pseudo-labeled negatives (NB02 query values): {len(pseudo_negatives):,}')
if pseudo_negatives:
    for p in pseudo_negatives[:5]:
        print(f"  '{p[:70]}'")
else:
    print('  (none — using synthetic only)')
print()

# ── Source 2: synthetic negatives at scale ───────────────────────
import random
random.seed(42)

base_ids     = [str(i) for i in range(1, 200)]
page_nums    = [str(i) for i in range(1, 51)]
sort_vals    = ['desc','asc','price desc','price asc','created desc','created asc',
                'newest','oldest','popular','relevance','default','rating desc']
filter_vals  = ['active','all','none','enabled','true','false','verified','new','used']
categories   = ['electronics','furniture','cars','jobs','services','real-estate',
                'mobiles','laptops','clothes','sports','books','cameras']
locations    = ['cairo','dubai','riyadh','amman','baghdad','tunis','casablanca',
                'kuwait','doha','muscat','abu-dhabi','beirut','algiers','damascus']
search_terms = ['laptop','mobile','car','apartment','samsung','iphone','toyota',
                'sofa','chair','camera','watch','shoes','dress','bicycle','tablet']

synthetics = []
synthetics += base_ids
synthetics += ['page ' + p for p in page_nums]
synthetics += sort_vals + filter_vals + categories + locations + search_terms
synthetics += [i + ' ' + s for i in base_ids[:50] for s in sort_vals[:3]]
synthetics += [t + ' ' + l for t in search_terms for l in locations[:5]]
synthetics += [c + ' ' + s for c in categories for s in sort_vals[:4]]
synthetics += [c + ' ' + s + ' ' + p
               for c in categories
               for s in sort_vals[:3]
               for p in page_nums[:5]]
synthetics += [str(i) for i in range(10000, 10200)]

SQL_KW = ["' ","select ","union ","insert ","drop ","delete "," or "," and "]
synthetics = list(dict.fromkeys(synthetics))
synthetics = [s for s in synthetics if not any(k in s.lower() for k in SQL_KW)]
print(f'Synthetic negatives: {len(synthetics):,}')

all_negatives = list(dict.fromkeys(pseudo_negatives + synthetics))
print(f'Total negatives (pseudo + synthetic, deduped): {len(all_negatives):,}')


Pseudo-labeled negatives (NB02 query values): 0
  (none — using synthetic only)

Synthetic negatives: 964
Total negatives (pseudo + synthetic, deduped): 964


## 5. Combine & Check Balance

In [5]:
all_queries = positives + all_negatives
all_labels  = [1] * len(positives) + [0] * len(all_negatives)

pos = int(sum(all_labels))
neg = len(all_labels) - pos
print(f'Positives : {pos:,}')
print(f'Negatives : {neg:,}  (pseudo={len(pseudo_negatives):,}  synthetic={len(synthetics):,})')
print(f'Total     : {len(all_queries):,}')
print(f'Ratio     : 1:{neg/pos:.2f}')
print()
print('Note: positive-dominant is expected — real FP behaviour is measured in NB04.')


Positives : 11,383
Negatives : 964  (pseudo=0  synthetic=964)
Total     : 12,347
Ratio     : 1:0.08

Note: positive-dominant is expected — real FP behaviour is measured in NB04.


## 6. Split First → Vectorize (Train-Only Fit)

In [6]:
q_train, q_test, y_train, y_test = train_test_split(
    all_queries, all_labels, test_size=0.2, random_state=42, stratify=all_labels)

vectorizer = CountVectorizer(analyzer='char', ngram_range=(1, 3))
X_train = hstack([vectorizer.fit_transform(q_train), build_symbol_matrix(q_train)])
X_test  = hstack([vectorizer.transform(q_test),      build_symbol_matrix(q_test)])
y_train = np.array(y_train)
y_test  = np.array(y_test)

print(f'Train  : {len(q_train):,}  pos={int((y_train==1).sum()):,}  neg={int((y_train==0).sum()):,}')
print(f'Test   : {len(q_test):,}   pos={int((y_test==1).sum()):,}   neg={int((y_test==0).sum()):,}')
print(f'Vocab  : {len(vectorizer.vocabulary_):,} + {len(SYMBOLS)} symbols = {X_train.shape[1]:,} features')

joblib.dump(vectorizer, 'results/models/03_vectorizer.pkl')
print('Saved: results/models/03_vectorizer.pkl')


Train  : 9,877  pos=9,106  neg=771
Test   : 2,470   pos=2,277   neg=193
Vocab  : 15,185 + 17 symbols = 15,202 features
Saved: results/models/03_vectorizer.pkl


## 7. Train Models

In [7]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SGD (log loss)':      SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'LinearSVC':           CalibratedClassifierCV(LinearSVC(max_iter=2000)),
    'Decision Tree':       DecisionTreeClassifier(),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results, cms, pr_data, trained = [], {}, {}, {}

for name, model in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    model.fit(X_train, y_train)

    t0     = time.perf_counter()
    y_pred = model.predict(X_test)
    lat_ms = (time.perf_counter() - t0) * 1000
    y_prob = model.predict_proba(X_test)[:, 1]

    f1    = f1_score(y_test, y_pred, zero_division=0)
    prauc = average_precision_score(y_test, y_prob)

    results.append({
        'Model':      name,
        'Accuracy':   round(accuracy_score(y_test, y_pred), 4),
        'Precision':  round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall':     round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-score':   round(f1, 4),
        'PR-AUC':     round(prauc, 4),
        'Latency_ms': round(lat_ms, 1),
    })
    cms[name]     = confusion_matrix(y_test, y_pred)
    pr_data[name] = precision_recall_curve(y_test, y_prob) + (prauc,)
    trained[name] = (model, y_prob)

    fname = name.replace(' ', '_').replace('(', '').replace(')', '').lower()
    joblib.dump(model, f'results/models/03_{fname}_model.pkl')
    print(f'F1={f1:.4f}  PR-AUC={prauc:.4f}  Lat={lat_ms:.1f}ms  saved')

results_df = pd.DataFrame(results)
results_df.to_csv('results/metrics/03_model_results.csv', index=False)
print()
print('=== TEST SET RESULTS ===')
print(results_df.to_string(index=False))


Training Logistic Regression... F1=0.9978  PR-AUC=1.0000  Lat=2.9ms  saved
Training SGD (log loss)... F1=0.9978  PR-AUC=1.0000  Lat=1.4ms  saved
Training LinearSVC... F1=0.9971  PR-AUC=0.9999  Lat=13.7ms  saved
Training Decision Tree... F1=0.9976  PR-AUC=0.9988  Lat=5.1ms  saved
Training Naive Bayes... F1=0.9895  PR-AUC=0.9999  Lat=4.1ms  saved
Training Random Forest... F1=0.9982  PR-AUC=1.0000  Lat=65.8ms  saved

=== TEST SET RESULTS ===
              Model  Accuracy  Precision  Recall  F1-score  PR-AUC  Latency_ms
Logistic Regression    0.9960     0.9996  0.9960    0.9978  1.0000         2.9
     SGD (log loss)    0.9960     0.9996  0.9960    0.9978  1.0000         1.4
          LinearSVC    0.9947     0.9991  0.9952    0.9971  0.9999        13.7
      Decision Tree    0.9955     0.9991  0.9960    0.9976  0.9988         5.1
        Naive Bayes    0.9806     0.9823  0.9969    0.9895  0.9999         4.1
      Random Forest    0.9968     0.9991  0.9974    0.9982  1.0000        65.8


## 8. Threshold Tuning — Corrected Direction

In [8]:
print('=== THRESHOLD TUNING (RF) ===')
print()
rf_probs       = trained['Random Forest'][1]
p_c, r_c, thr, _ = pr_data['Random Forest']

# Scan from HIGH threshold downward — guarantees T_high > T_low
pairs = sorted(zip(thr, p_c[:-1], r_c[:-1]), key=lambda x: -x[0])

t_high = p_high = r_high = None
for t, p, r in pairs:
    if p >= 0.995:
        t_high, p_high, r_high = round(float(t),3), round(float(p),4), round(float(r),4)
        break

t_low = p_low = r_low = None
for t, p, r in pairs:
    if r >= 0.95:
        t_low, p_low, r_low = round(float(t),3), round(float(p),4), round(float(r),4)
        break

assert t_high is not None, "No threshold found with precision >= 0.99"
assert t_low  is not None, "No threshold found with recall >= 0.95"
assert t_high > t_low, f"T_high ({t_high}) must be > T_low ({t_low})"

print(f'T_high (highest t where prec>=0.99) : {t_high}  prec={p_high}  rec={r_high}')
print(f'T_low  (highest t where rec>=0.95)  : {t_low}   prec={p_low}   rec={r_low}')
print(f'T_high > T_low : {t_high} > {t_low}  ✅')

th_dict = {
    't_high': t_high, 'p_high': p_high, 'r_high': r_high,
    't_low':  t_low,  'p_low':  p_low,  'r_low':  r_low,
}
with open('results/models/03_thresholds.json', 'w') as f:
    json.dump(th_dict, f, indent=2)
print('Saved: results/models/03_thresholds.json')


=== THRESHOLD TUNING (RF) ===

T_high (highest t where prec>=0.99) : 1.0  prec=1.0  rec=0.935
T_low  (highest t where rec>=0.95)  : 0.99   prec=1.0   rec=0.9728
T_high > T_low : 1.0 > 0.99  ✅
Saved: results/models/03_thresholds.json


## 9. 5-Fold Cross-Validation

In [9]:
print('Running 5-fold CV (vectorizer refit per fold)...')
print()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
qa, la = np.array(all_queries), np.array(all_labels)

cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'SGD (log loss)':      SGDClassifier(loss='log_loss', max_iter=1000, random_state=42),
    'LinearSVC':           CalibratedClassifierCV(LinearSVC(max_iter=2000)),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

cv_rows = []
for name, model in cv_models.items():
    fold_f1, fold_pr = [], []
    for tr_i, va_i in skf.split(qa, la):
        v   = CountVectorizer(analyzer='char', ngram_range=(1, 3))
        Xtr = hstack([v.fit_transform(qa[tr_i].tolist()), build_symbol_matrix(qa[tr_i].tolist())])
        Xva = hstack([v.transform(qa[va_i].tolist()),     build_symbol_matrix(qa[va_i].tolist())])
        model.fit(Xtr, la[tr_i])
        yp   = model.predict(Xva)
        yprb = model.predict_proba(Xva)[:, 1]
        fold_f1.append(f1_score(la[va_i], yp, zero_division=0))
        fold_pr.append(average_precision_score(la[va_i], yprb))

    cv_rows.append({
        'Model':         name,
        'CV_F1_mean':    round(np.mean(fold_f1), 4),
        'CV_F1_std':     round(np.std(fold_f1),  4),
        'CV_PRAUC_mean': round(np.mean(fold_pr),  4),
        'CV_PRAUC_std':  round(np.std(fold_pr),   4),
    })
    print(f'  {name:25s}: F1={np.mean(fold_f1):.4f}±{np.std(fold_f1):.4f}'
          f'  PR-AUC={np.mean(fold_pr):.4f}±{np.std(fold_pr):.4f}')

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv('results/metrics/03_cv_results.csv', index=False)
print()
print(cv_df.to_string(index=False))


Running 5-fold CV (vectorizer refit per fold)...

  Logistic Regression      : F1=0.9973±0.0008  PR-AUC=1.0000±0.0000
  SGD (log loss)           : F1=0.9974±0.0007  PR-AUC=1.0000±0.0000
  LinearSVC                : F1=0.9967±0.0013  PR-AUC=0.9999±0.0001
  Naive Bayes              : F1=0.9898±0.0018  PR-AUC=0.9999±0.0000
  Random Forest            : F1=0.9982±0.0006  PR-AUC=1.0000±0.0000

              Model  CV_F1_mean  CV_F1_std  CV_PRAUC_mean  CV_PRAUC_std
Logistic Regression      0.9973     0.0008         1.0000        0.0000
     SGD (log loss)      0.9974     0.0007         1.0000        0.0000
          LinearSVC      0.9967     0.0013         0.9999        0.0001
        Naive Bayes      0.9898     0.0018         0.9999        0.0000
      Random Forest      0.9982     0.0006         1.0000        0.0000


## 10. Benchmark — 50 Attacks + 50 Benign

In [10]:
print('=== BENCHMARK: 50 attacks + 50 benign ===')
print()
np.random.seed(99)
rf_model = trained['Random Forest'][0]

# 50 attacks
attack_pool  = df_clean[df_clean['Label'] == 1]['Query'].tolist()
bench_atk_qv = []
for i, q in enumerate(np.random.choice(attack_pool, 50, replace=False)):
    url = TEMPLATES[i % len(TEMPLATES)].format(
        urllib.parse.quote(str(q), safe="'\"=;-/*+()|<>\\#@ "))
    val = extract_query_values(url)
    if val:
        bench_atk_qv.append(val)

# 50 benign
n_pseudo      = min(25, len(pseudo_negatives))
bench_ben_qv  = pseudo_negatives[:n_pseudo] + synthetics[:(50 - n_pseudo)]
bench_ben_qv  = bench_ben_qv[:50]

def score_rf(qv_list):
    preds, probs = [], []
    for qv in qv_list:
        Xq   = hstack([vectorizer.transform([qv]), build_symbol_matrix([qv])])
        preds.append(int(rf_model.predict(Xq)[0]))
        probs.append(float(rf_model.predict_proba(Xq)[0][1]))
    return preds, probs

atk_preds, atk_probs = score_rf(bench_atk_qv)
ben_preds, ben_probs = score_rf(bench_ben_qv)

print(f'Attack detection (first 10 of {len(bench_atk_qv)}):')
for qv, pred, prob in zip(bench_atk_qv[:10], atk_preds[:10], atk_probs[:10]):
    print(f'  {"✅" if pred==1 else "❌"} prob={prob:.3f}  {repr(qv[:60])}')

print(f'\nBenign FP check (first 10 of {len(bench_ben_qv)}):')
for qv, pred, prob in zip(bench_ben_qv[:10], ben_preds[:10], ben_probs[:10]):
    print(f'  {"❌ FP" if pred==1 else "✅ OK"} prob={prob:.3f}  {repr(qv[:60])}')

det = sum(atk_preds)
fps = sum(ben_preds)
print(f'\nBENCHMARK RESULT:')
print(f'  Attack detection : {det}/{len(bench_atk_qv)}  ({det/len(bench_atk_qv)*100:.0f}%)')
print(f'  Benign FPs       : {fps}/{len(bench_ben_qv)}  ({fps/len(bench_ben_qv)*100:.0f}%)')
if det + fps > 0:
    print(f'  Benchmark prec   : {det/(det+fps):.4f}')


=== BENCHMARK: 50 attacks + 50 benign ===

Attack detection (first 10 of 50):
  ✅ prob=1.000  'or a  =  a--'
  ✅ prob=1.000  '1"  )   and 7756  =  dbms_utility.sqlid_to_sqlhash   (    ( '
  ✅ prob=1.000  "1'   (  select rdwb where 2498  =  2498 and 3429  =  7639--"
  ✅ prob=1.000  '-2374  )   or 2724 in    (    (   char  (  113  )   char  ( '
  ✅ prob=1.000  'Ã½ or 1  =  1 --'
  ✅ prob=1.000  'select dbms_pipe.receive_message  (  chr  (  66  )  ||chr  ('
  ✅ prob=1.000  "-2733'   )    )     )   or 4144  =    (  select upper  (  xm"
  ✅ prob=1.000  "-1055' union all select 7758,7758,7758,7758,7758,7758--"
  ✅ prob=1.000  '1,  (  select   (  case when   (  5885  =  1825  )   then 1 '
  ✅ prob=1.000  '-5082   )    )    union all select 4013,4013,4013,4013,4013,'

Benign FP check (first 10 of 50):
  ✅ OK prob=0.435  '1'
  ✅ OK prob=0.130  '2'
  ✅ OK prob=0.047  '3'
  ✅ OK prob=0.053  '4'
  ✅ OK prob=0.141  '5'
  ✅ OK prob=0.050  '6'
  ✅ OK prob=0.065  '7'
  ✅ OK prob=0.040  '8'
  ✅ OK prob

## 11. Visualisations

In [11]:
fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
for ax, (name, cm) in zip(axes, cms.items()):
    ConfusionMatrixDisplay(cm, display_labels=['legit', 'attack']).plot(
        cmap=plt.cm.Blues, ax=ax, colorbar=False)
    ax.set_title(name, fontsize=8)
plt.suptitle('Confusion Matrices — Way 3 Test Set', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('results/figures/03_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.close()

fig, ax = plt.subplots(figsize=(9, 6))
for name, (p, r, t, auc) in pr_data.items():
    ax.plot(r, p, label=f'{name} (AUC={auc:.4f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('PR Curves — Way 3')
ax.legend(loc='lower left', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/figures/03_pr_curves.png', dpi=150)
plt.close()

print('Saved: 03_confusion_matrices.png  03_pr_curves.png')


Saved: 03_confusion_matrices.png  03_pr_curves.png


## 12. Summary

In [12]:
th = json.load(open('results/models/03_thresholds.json'))
print('=' * 65)
print('NOTEBOOK 03 — COMPLETE (Way 3, fixed negatives)')
print('=' * 65)
print(f'Positives : {len(positives):,}')
print(f'Negatives : {len(all_negatives):,}'
      f'  (pseudo={len(pseudo_negatives):,}  synthetic={len(synthetics):,})')
print(f'Total     : {len(all_queries):,}')
print()
print('TEST SET:')
print(results_df[['Model', 'Precision', 'Recall', 'F1-score', 'PR-AUC', 'Latency_ms']].to_string(index=False))
print()
print('5-FOLD CV:')
print(cv_df[['Model', 'CV_F1_mean', 'CV_F1_std', 'CV_PRAUC_mean']].to_string(index=False))
print()
print(f'THRESHOLDS (RF):')
print(f'  T_high = {th["t_high"]}  precision={th["p_high"]}  recall={th["r_high"]}')
print(f'  T_low  = {th["t_low"]}   precision={th["p_low"]}   recall={th["r_low"]}')
print(f'  T_high > T_low : {th["t_high"]} > {th["t_low"]}  ✅')
print()
print('NEXT: Notebook 04 — Re-evaluate on labeled_access.log (Way 3 inference)')


NOTEBOOK 03 — COMPLETE (Way 3, fixed negatives)
Positives : 11,383
Negatives : 964  (pseudo=0  synthetic=964)
Total     : 12,347

TEST SET:
              Model  Precision  Recall  F1-score  PR-AUC  Latency_ms
Logistic Regression     0.9996  0.9960    0.9978  1.0000         2.9
     SGD (log loss)     0.9996  0.9960    0.9978  1.0000         1.4
          LinearSVC     0.9991  0.9952    0.9971  0.9999        13.7
      Decision Tree     0.9991  0.9960    0.9976  0.9988         5.1
        Naive Bayes     0.9823  0.9969    0.9895  0.9999         4.1
      Random Forest     0.9991  0.9974    0.9982  1.0000        65.8

5-FOLD CV:
              Model  CV_F1_mean  CV_F1_std  CV_PRAUC_mean
Logistic Regression      0.9973     0.0008         1.0000
     SGD (log loss)      0.9974     0.0007         1.0000
          LinearSVC      0.9967     0.0013         0.9999
        Naive Bayes      0.9898     0.0018         0.9999
      Random Forest      0.9982     0.0006         1.0000

THRESHOLDS (RF):

C:\Users\muham\AppData\Local\Temp\ipykernel_22896\3565559503.py:1: ResourceWarning: unclosed file <_io.TextIOWrapper name='results/models/03_thresholds.json' mode='r' encoding='cp1252'>
  th = json.load(open('results/models/03_thresholds.json'))
